# MetaSeg on Oasis Brain MRI data


## 1. Requirements

1 - Upload ``neurite-oasis.v1.0.tar`` to Google Drive

2 - Upload ``oasis_splits_3d.json`` in content

3 - Create a folder ``dumps`` the with weights.


In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!tar -xvf /content/drive/MyDrive/neurite-oasis.v1.0.tar

Streaming output truncated to the last 5000 lines.
OASIS_OAS1_0032_MR1/orig.nii.gz
OASIS_OAS1_0032_MR1/seg4.nii.gz
OASIS_OAS1_0032_MR1/slice_norm.nii.gz
OASIS_OAS1_0033_MR1/
OASIS_OAS1_0033_MR1/seg35.nii.gz
OASIS_OAS1_0033_MR1/orig.nii.gz
OASIS_OAS1_0033_MR1/slice_norm.nii.gz
OASIS_OAS1_0033_MR1/seg4.nii.gz
OASIS_OAS1_0033_MR1/aligned_seg4.nii.gz
OASIS_OAS1_0033_MR1/aligned_seg35.nii.gz
OASIS_OAS1_0033_MR1/aligned_orig.nii.gz
OASIS_OAS1_0033_MR1/slice_seg24.nii.gz
OASIS_OAS1_0033_MR1/aligned_norm.nii.gz
OASIS_OAS1_0033_MR1/norm.nii.gz
OASIS_OAS1_0033_MR1/slice_seg4.nii.gz
OASIS_OAS1_0033_MR1/slice_orig.nii.gz
OASIS_OAS1_0034_MR1/
OASIS_OAS1_0034_MR1/aligned_seg35.nii.gz
OASIS_OAS1_0034_MR1/norm.nii.gz
OASIS_OAS1_0034_MR1/slice_norm.nii.gz
OASIS_OAS1_0034_MR1/aligned_orig.nii.gz
OASIS_OAS1_0034_MR1/aligned_seg4.nii.gz
OASIS_OAS1_0034_MR1/seg35.nii.gz
OASIS_OAS1_0034_MR1/aligned_norm.nii.gz
OASIS_OAS1_0034_MR1/slice_seg24.nii.gz
OASIS_OAS1_0034_MR1/slice_seg4.nii.gz
OASIS_OAS1_0034_MR1/s

In [ ]:
!pip install ipdb
!pip install monai
!pip install torchmetrics
!pip install torchinfo



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 63.8 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.utils
import torch.utils.data as torch_data

import numpy as np
from copy import deepcopy
from tqdm.autonotebook import tqdm
from collections import OrderedDict

import logging
import os, os.path as osp
from matplotlib import pyplot as plt

import pdb
import math

import time
import glob
import ipdb
from ipdb import set_trace as debug
from pdb import set_trace as debug


from scipy.spatial.distance import dice, jaccard
from torchmetrics.functional.segmentation import generalized_dice_score

import nibabel as nib
import json

#import libINR.utils.coords

import sys
from datetime import datetime
import cv2
import skimage.filters, skimage.transform


import SimpleITK as sitk


from copy import deepcopy
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap


/tmp/ipython-input-3352/367815668.py:9: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## 2. Models

In [ ]:
class BaseINR(nn.Module):
    def __init__(self, device=torch.device("cuda" if torch.cuda.is_available() else "cpu")):
        super(BaseINR, self).__init__()

        self.model = None
        self.device = device
        self.optimizer = None
        self.loss_function = nn.MSELoss().to(self.device)


    def compile(self, optimizer_name="adam", learning_rate=1e-4, scheduler=None,):
        if optimizer_name == "adam":
            self.optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)
        elif optimizer_name == "sgd":
            self.optimizer = torch.optim.SGD(self.parameters(), lr=learning_rate)
        else:
            raise ValueError("Optimizer not supported")

        if scheduler is not None:
            self.scheduler = scheduler
        else:
            self.scheduler = None

    def set_loss_function(self, loss_function):
        self.loss_function = loss_function.to(self.device)

    def fit(self, input_vectors, signal, epochs=1000, report_metrics=None, disable_tqdm=False):
        input_vectors = input_vectors.to(self.device)
        signal = signal.to(self.device)
        pbar = tqdm(range(epochs)) if not disable_tqdm else range(epochs)

        compute_metrics_for_training = False

        if report_metrics is not None:
            compute_metrics_for_training = True

        for epoch in pbar:
            self.optimizer.zero_grad()
            model_output = self(input_vectors)
            loss = self.loss_function(model_output, signal)
            loss.backward()
            self.optimizer.step()

            if not disable_tqdm:
                metrics_string = ''
                pbar.set_description(f"Epoch: {epoch}/{epochs}. Loss: {loss.item():.6f} {metrics_string if report_metrics is not None else ''}")
            if self.scheduler is not None:
                self.scheduler.step()

    def predict(self, input_vectors):
        with torch.no_grad():
            input_vectors = input_vectors.to(self.device)
            return self(input_vectors).cpu().numpy()

    def predict_allow_gradients(self, input_vectors):
        input_vectors = input_vectors.to(self.device)
        return self(input_vectors)


    def get_weights(self):
        return self.state_dict()

    def copy_from_model(self, model):
        self.load_state_dict(deepcopy({ k:v.detach().clone() for k,v in model.state_dict().items()} ))

In [ ]:
class INRMetaLearner():
    def __init__(self, model, inner_steps, config={}, custom_loss_fn=None, outer_optimizer='adam', inner_loop_loss_fn=None):
        super(INRMetaLearner, self).__init__()
        self.model = model
        self.inner_steps = inner_steps
        self.config = config

        self.model_params = OrderedDict({
            k: v.clone().detach().requires_grad_() for k, v in self.model.named_parameters()
        })

        self.outer_optimizer = outer_optimizer
        self.configure_optimizers()

        self.loss_fn = custom_loss_fn if custom_loss_fn is not None else self.loss_fn_mse
        self.inner_loop_loss_fn = inner_loop_loss_fn

    def configure_optimizers(self):
        lr = self.config.get("outer_lr", 1e-4)
        if self.outer_optimizer == 'adam':
            self.opt_thetas = torch.optim.Adam(self.model_params.values(), lr=lr)
        else:
            self.opt_thetas = torch.optim.SGD(self.model_params.values(), lr=lr)

    def get_parameters(self, copy=True):
        return OrderedDict({
            k: v.clone().detach() if copy else v for k, v in self.model_params.items()
        })

    def get_inr_parameters(self, key_prefix="inr", copy=True):
        return OrderedDict({
            k: v.clone().detach() if copy else v for k, v in self.model_params.items() if key_prefix in k
        })

    def get_segmentation_parameters(self, key_prefix='segmentation', copy=True):
        return OrderedDict({
            k: v.clone().detach() if copy else v for k, v in self.model_params.items() if key_prefix in k
        })

    def set_parameters(self, params):
        self.model_params.update(params)

    def mse_loss(self, x, y):
        return nn.functional.mse_loss(x, y)

    def loss_fn_mse(self, data_packet):
        mse_loss = self.mse_loss(data_packet['output']['inr_output'], data_packet['gt'])
        return mse_loss, {'mse_loss': float(mse_loss)}

    def squeeze_output(self, output, gt):
        if isinstance(output, dict):
            for k, v in output.items():
                if v.dim() != gt.dim():
                    output[k] = v.squeeze(1)
        elif isinstance(output, torch.Tensor):
            if output.dim() != gt.dim():
                output = output.squeeze(1)
        return output

    def adam_update(self, params, grads, opt_step, adam_m, adam_v, lr=1e-4, beta1=0.9, beta2=0.999, epsilon=1e-8):
        updated_params = OrderedDict()
        for ((k,p), g) in zip(params.items(), grads):
            adam_m[k] = beta1 * adam_m[k] + (1 - beta1) * g
            adam_v[k] = beta2 * adam_v[k] + (1 - beta2) * (g**2)

            m_hat = adam_m[k] / (1 - beta1 ** (opt_step))
            v_hat = adam_v[k] / (1 - beta2 ** (opt_step))
            v_hat = torch.clamp(v_hat, min=1e-12)
            updated_p = p - lr * m_hat / (torch.sqrt(v_hat) + epsilon)

            updated_params[k] = updated_p.requires_grad_()

        return updated_params, adam_m, adam_v

    def inner_loop(self, coords, data_packet, create_graph=True):
        gt = data_packet['gt']
        params = OrderedDict((k, v.clone()) for k, v in self.model_params.items())

        adam_m = {k: torch.zeros_like(v) for k, v in params.items()}
        adam_v = {k: torch.zeros_like(v) for k, v in params.items()}

        for i in range(self.inner_steps):
            output = torch.func.functional_call(self.model, params, coords)
            output = self.squeeze_output(output, gt)
            data_packet['output'] = output

            if self.inner_loop_loss_fn is not None:
                loss, _ = self.inner_loop_loss_fn(data_packet)
            else:
                loss, _ = self.loss_fn(data_packet)
            if torch.isnan(loss).any():
                print("Loss is NaN, skipping this step")
                break
            grads = torch.autograd.grad(loss, params.values(), create_graph=True)

            lr = self.config.get("inner_lr", 1e-4)
            params, adam_m, adam_v = self.adam_update(params, grads, i+1, adam_m, adam_v, lr=lr)
            logging.info(f"Inner step: {i+1}/{self.inner_steps}, Loss: {loss.item():.6f}")
        return params

    def outer_loop(self, coords, data_packet):
        self.opt_thetas.zero_grad()
        gt = data_packet['gt']
        adapted_params = self.inner_loop(coords, data_packet, create_graph=True)

        output = torch.func.functional_call(self.model, adapted_params, coords)
        output = self.squeeze_output(output, gt)
        data_packet['output'] = output
        loss, loss_info = self.loss_fn(data_packet)

        loss.backward()
        self.opt_thetas.step()
        return loss, loss_info

    def forward(self, coords, data_packet):
        return self.outer_loop(coords, data_packet)

    def render_inner_loop(self, coords, gt, inner_loop_steps=1):
        model_copy = deepcopy(self.model)
        params = OrderedDict({
            k: v.clone().detach().requires_grad_(True) for k, v in self.model_params.items() if "inr" in k
        })
        opt_render = torch.optim.Adam(params.values(), lr=self.config.get("inner_lr", 1e-4))

        for _ in range(inner_loop_steps):
            opt_render.zero_grad()
            output = torch.func.functional_call(model_copy, params, coords)
            loss = nn.functional.mse_loss(output['inr_output'], gt)
            loss.backward()
            opt_render.step()

        output = self.squeeze_output(output, gt)
        return {'output': output}

In [ ]:

try:
    BaseINR
except NameError:
    class BaseINR(nn.Module):
        def __init__(self):
            super().__init__()

        def positional_encoding(self, coords):
            return coords


class ResidualDualFreqSineLayer(nn.Module):
    """
    Residual Dual-Frequency Gated SIREN Layer
    -----------------------------------------
    y = g * sin(omega_low * Wx) + (1-g) * sin(omega_high * Wx) + Res(x)

    Benefits:
    - Learns local frequency mixture (coarse + fine)
    - Residual skip improves optimization stability
    - Learnable omega supports data-adaptive spectral tuning
    """
    def __init__(
        self,
        in_features,
        out_features,
        bias=True,
        is_first=False,
        omega_0=30.0,
        scale=10.0,
        init_weights=True,
        low_freq_ratio=0.5,
        high_freq_ratio=2.0,
        learnable_omega=True,
    ):
        super().__init__()
        self.is_first = is_first
        self.in_features = in_features
        self.out_features = out_features

        self.linear = nn.Linear(in_features, out_features, bias=bias)

        if learnable_omega:
            self.omega_0 = nn.Parameter(torch.tensor(float(omega_0)))
        else:
            self.register_buffer("omega_0", torch.tensor(float(omega_0)))

        self.register_buffer("low_ratio", torch.tensor(float(low_freq_ratio)))
        self.register_buffer("high_ratio", torch.tensor(float(high_freq_ratio)))

        self.gate = nn.Linear(out_features, out_features, bias=True)

        if in_features == out_features:
            self.res_proj = nn.Identity()
        else:
            self.res_proj = nn.Linear(in_features, out_features, bias=False)

        if init_weights:
            self.init_weights()

    def init_weights(self):
        with torch.no_grad():
            if self.is_first:
                self.linear.weight.uniform_(-1 / self.in_features, 1 / self.in_features)
            else:
                om = float(torch.clamp(self.omega_0.detach(), min=1e-6).item())
                bound = np.sqrt(6 / self.in_features) / om
                self.linear.weight.uniform_(-bound, bound)

            self.gate.weight.zero_()
            self.gate.bias.zero_()

            if isinstance(self.res_proj, nn.Linear):
                nn.init.xavier_uniform_(self.res_proj.weight)

    def forward(self, x):
        z = self.linear(x)

        om = torch.clamp(self.omega_0, min=1e-6)
        y_low = torch.sin((om * self.low_ratio) * z)
        y_high = torch.sin((om * self.high_ratio) * z)

        g = torch.sigmoid(self.gate(z))
        y = g * y_low + (1.0 - g) * y_high

        return y + self.res_proj(x)


class INR(BaseINR):
    def __init__(
        self,
        in_features,
        hidden_features,
        hidden_layers,
        out_features,
        outermost_linear=True,
        first_omega_0=30,
        hidden_omega_0=30.0,
        scale=10.0,
        pos_encode=False,
        sidelength=512,
        fn_samples=None,
        use_nyquist=True,

        low_freq_ratio=0.5,
        high_freq_ratio=2.0,
        learnable_omega=True,
    ):
        super().__init__()
        self.pos_encode = pos_encode

        self.nonlin = ResidualDualFreqSineLayer

        self.net_w_layers = []
        self.net_w_layers.append(
            self.nonlin(
                in_features,
                hidden_features,
                is_first=True,
                omega_0=first_omega_0,
                scale=scale,
                low_freq_ratio=low_freq_ratio,
                high_freq_ratio=high_freq_ratio,
                learnable_omega=learnable_omega,
            )
        )

        for _ in range(hidden_layers):
            self.net_w_layers.append(
                self.nonlin(
                    hidden_features,
                    hidden_features,
                    is_first=False,
                    omega_0=hidden_omega_0,
                    scale=scale,
                    low_freq_ratio=low_freq_ratio,
                    high_freq_ratio=high_freq_ratio,
                    learnable_omega=learnable_omega,
                )
            )

        if outermost_linear:
            dtype = torch.float
            final_linear = nn.Linear(hidden_features, out_features, dtype=dtype)

            with torch.no_grad():
                const = np.sqrt(6 / hidden_features) / max(hidden_omega_0, 1e-12)
                final_linear.weight.uniform_(-const, const)

            self.net_w_layers.append(final_linear)
        else:
            self.net_w_layers.append(
                self.nonlin(
                    hidden_features,
                    out_features,
                    is_first=False,
                    omega_0=hidden_omega_0,
                    scale=scale,
                    low_freq_ratio=low_freq_ratio,
                    high_freq_ratio=high_freq_ratio,
                    learnable_omega=learnable_omega,
                )
            )

        self.net = nn.Sequential(*self.net_w_layers)

    def forward(self, coords):
        if self.pos_encode:
            coords = self.positional_encoding(coords)

        output = self.net(coords)
        return output

    def forward_w_features(self, coords):
        if self.pos_encode:
            coords = self.positional_encoding(coords)

        features = []
        x = coords.clone()
        for layer in self.net_w_layers:
            x = layer(x)
            features.append(x)

        output = x.clone()
        return output, features

    def load_weights(self, weights):
        self.load_state_dict(weights)


class SegmentationModel(nn.Module):
    def __init__(self, inr_config, segmentation_config):
        super(SegmentationModel, self).__init__()

        seg_layers = [
            nn.Linear(inr_config['hidden_features'], segmentation_config['hidden_features'][0]),
            nn.LeakyReLU()
        ]

        for i in range(1, len(segmentation_config['hidden_features'])):
            seg_layers.append(
                nn.Linear(
                    segmentation_config['hidden_features'][i - 1],
                    segmentation_config['hidden_features'][i]
                )
            )
            seg_layers.append(nn.LeakyReLU())

        seg_layers.append(
            nn.Linear(
                segmentation_config['hidden_features'][-1],
                segmentation_config['output_features']
            )
        )

        self.segmentation_head = nn.Sequential(*seg_layers)

    def forward(self, features):
        return self.segmentation_head(features)


class SirenSegINR(nn.Module):
    def __init__(self, inr_type, inr_config, segmentation_config, normalize_features=False):
        super(SirenSegINR, self).__init__()

        self.inr = INR(**inr_config)
        self.normalize_features = normalize_features
        self.segmentation_head = SegmentationModel(inr_config, segmentation_config)

        print(self.inr)
        print(self.segmentation_head)

    def forward(self, coords):
        inr_output, inr_features = self.inr.forward_w_features(coords)
        penultimate_features = inr_features[-2]

        if self.normalize_features:
            penultimate_features = F.normalize(penultimate_features, dim=-1)

        segmentation_output = self.segmentation_head(penultimate_features)

        return {
            'inr_output': inr_output,
            'inr_features': penultimate_features,
            'segmentation_output': segmentation_output
        }

    def set_weights(self, weights):
        self.load_state_dict(weights)

    def get_weights(self):
        return self.state_dict()


class SirenINR(nn.Module):
    def __init__(self, inr_type, inr_config, segmentation_config, normalize_features=False):
        super(SirenINR, self).__init__()

        self.inr = INR(**inr_config)
        self.normalize_features = normalize_features

        print(self.inr)

    def forward(self, coords):

        inr_output, inr_features = self.inr.forward_w_features(coords)
        return {'inr_output': inr_output}

    def set_weights(self, weights):
        self.load_state_dict(weights)

    def get_weights(self):
        return self.state_dict()


## 3. Metrics and Losses



In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, alpha=1.0, smoothing = 1e-5):
        super(DiceLoss, self).__init__()
        self.smoothing = smoothing
        self.alpha = alpha

    def forward(self, prediction, target):
        pred_probs = nn.functional.softmax(prediction, dim=1)
        target_probs = target

        intersection = torch.sum(pred_probs * target_probs)
        dice = (2. * intersection + self.smoothing) / (torch.sum(pred_probs) + torch.sum(target_probs) + self.smoothing)

        return (1 - dice) * self.alpha

class MSELoss(nn.Module):
    def __init__(self, alpha=1.0, device=torch.device('cuda'), reduction='mean', zero_weight=0.01):
        super(MSELoss, self).__init__()
        self.alpha = alpha
        self.reduction = reduction
        self.zero_weight = zero_weight
        if self.reduction in ['sum' , 'mean']:
            self.mse_loss = nn.MSELoss().to(device)
        else:
            self.mse_loss = nn.MSELoss(reduction='none').to(device)

    def forward(self, data_packet):
        gt_img = data_packet['gt']
        seg_int = data_packet['seg_integer']
        inr_output = data_packet['output']['inr_output']
        if self.reduction in ['mean', 'sum']:
            mse_loss = self.mse_loss(inr_output, gt_img)
        else:
            weights = torch.where(seg_int == 0, self.zero_weight, 1.0)
            mse_loss_main = weights*torch.abs(inr_output - gt_img)**2
            mse_loss = torch.sum(mse_loss_main)/torch.sum(weights)
        return mse_loss * self.alpha

class FocalSemanticLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, device=torch.device('cuda')):
        super(FocalSemanticLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce_loss = CrossEntropyLoss().to(device)

    def forward(self, data_packet):
        gt_seg = data_packet['seg']
        segmentation_output = data_packet['output']['segmentation_output']
        seg_integers = torch.argmax(gt_seg, dim=-1)
        seg_prob_log = nn.functional.log_softmax(segmentation_output, dim=-1)

        grouped_seg_log_probs = torch.gather(seg_prob_log,
                                                dim=-1,
                                                index=seg_integers.unsqueeze(-1)).reshape(-1)

        scaled_grouped_probs = (1 - torch.exp(grouped_seg_log_probs)) ** self.gamma
        loss = -1 * scaled_grouped_probs * grouped_seg_log_probs

        return loss.mean()


class CrossEntropyLoss(nn.Module):
    def __init__(self, alpha=1.0, reduction='mean', device=torch.device('cuda')):
        super(CrossEntropyLoss, self).__init__()
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss(reduction=reduction).to(device)

    def forward(self, data_packet):

        gt_seg = data_packet['seg']
        segmentation_output = data_packet['output']['segmentation_output']
        seg_probs = segmentation_output
        ce_loss = self.ce_loss(seg_probs.permute(0,2,1), gt_seg.permute(0,2,1))

        return ce_loss * self.alpha

class LossFunction(nn.Module):
    def __init__(self, loss_funcs={'mse': MSELoss(alpha=1.0)},):
        super(LossFunction, self).__init__()
        self.loss_funcs = loss_funcs

    def forward(self, data_packet):
        total_loss = 0.0
        loss_dict = {}
        for key, loss_func in self.loss_funcs.items():
            loss_val = loss_func(data_packet)
            total_loss +=  loss_val
            loss_dict[key] = float(loss_val.item())

        return total_loss, loss_dict

In [ ]:
def multiclass_dice_score(segmentation, gt, num_classes):
    assert segmentation.shape == gt.shape, f"Segmentation and ground truth must have the same shape. {segmentation.shape=} {gt.shape=}"
    assert segmentation.shape[-1] == num_classes, f"Segmentation must have the same number of classes as the number of classes. {segmentation.shape=}  {num_classes=} "
    if segmentation.shape[0] != 1:
        segmentation = segmentation[None,...]
    if gt.shape[0] != 1:
        gt = gt[None,...]

    segmentation = segmentation.permute(0, 3,1,2)
    gt = gt.permute(0, 3,1,2)

    dice_scores_per_class = generalized_dice_score(
        segmentation,
        gt,
        include_background=True,
        num_classes=num_classes,
        weight_type='simple',
    )

    return dice_scores_per_class



def multiclass_dice_score_3d(segmentation, gt, num_classes):
    assert segmentation.shape == gt.shape, f"Segmentation and ground truth must have the same shape. {segmentation.shape=} {gt.shape=}"
    assert segmentation.shape[-1] == num_classes, f"Segmentation must have the same number of classes as the number of classes. {segmentation.shape=}  {num_classes=} "
    if segmentation.shape[0] != 1:
        segmentation = segmentation[None,...]
    if gt.shape[0] != 1:
        gt = gt[None,...]

    segmentation = segmentation.permute(0,4,1,2,3)
    gt = gt.permute(0,4,1,2,3)

    dice_scores_per_class = generalized_dice_score(
        segmentation,
        gt,
        include_background=True,
        num_classes=num_classes,
        weight_type='simple',
    )

    return dice_scores_per_class

import skimage.metrics

def psnr2(x, y):
    return skimage.metrics.peak_signal_noise_ratio(x, y, data_range=1)

import torch
def dice_simple(pred, gt_onehot, num_classes=5):
    pred_probs = torch.nn.functional.softmax(pred, -1)
    pred_probs_logit = torch.nn.functional.one_hot(torch.argmax(pred_probs, dim=-1), num_classes=num_classes)
    score = generalized_dice_score(pred_probs_logit.permute(0,3,1,2), gt_onehot.permute(0,3,1,2), num_classes=num_classes, weight_type='simple')
    return score.mean()


def multiclass_dice_score_3d_for_flat(pred_onehot, gt_onehot, num_classes):
    pred_reshaped = pred_onehot.permute(0, 2, 1)
    gt_reshaped = gt_onehot.permute(0, 2, 1)
    dice_scores_per_class = generalized_dice_score(
        pred_reshaped,
        gt_reshaped,
        include_background=True,
        num_classes=num_classes,
        weight_type='simple',
    )
    return dice_scores_per_class

## 4. Utils

In [ ]:
def show_image_subplot(images : list [np.ndarray], num_rows:int, num_cols :int , titles : list[str] = [], axis:str ='off' , dpi:int = 100) -> None:
    assert num_rows > 1 or num_cols > 1 , "Please ensure that you have more than 1 row or col"
    assert len(images) == int(num_rows*num_cols), "Please ensure that number of images provided match rows x cols product"
    titles = titles + ["No Title" for _ in range(len(images)-len(titles))] if len(titles) < len(images) else titles[:len(images)]
    fig, axs = plt.subplots(nrows=num_rows, ncols=num_cols, dpi=dpi)
    axs_flat = axs.flatten()
    for (_ax, _im, _title) in zip(axs_flat, images, titles):
        _ax.imshow(_im)
        _ax.axis(axis)
        _ax.set_title(_title)
    plt.tight_layout()
    plt.show()

def convert_tensor_to_onehot(x, num_classes):
    x = torch.from_numpy(x) if isinstance(x, np.ndarray) else x
    onehot = torch.zeros(size=(x.shape[0], x.shape[1], num_classes), dtype=torch.int32)
    for c in range(num_classes):
        onehot[..., c] = (x == c)
    return onehot

In [ ]:
def plot_result_row(imgs, titles, save=None, show=False):
    fig, axs = plt.subplots(1, len(imgs))
    for  (_ax, img, title) in zip(axs.flatten(), imgs, titles):
        _ax.axis('off')
        _ax.imshow(img)
        _ax.set_title(title)
    plt.tight_layout()

    if save is not None:
        plt.savefig(save)
    if show:
        plt.show()



In [ ]:
def get_coords2d(H, W):

    x = torch.linspace(-1, 1, W)
    y = torch.linspace(-1, 1, H)
    xx, yy = torch.meshgrid(x, y)
    coords = torch.hstack([xx.reshape(-1, 1), yy.reshape(-1, 1)])
    return coords

## 5. DataLoaders

In [ ]:
def resize_image(image, resolution, mode='nearest'):
    h, w = image.shape[:2]
    if h > w:
        padd_left = (h - w) // 2
        padd_right = h - w - padd_left
        image = np.pad(image, ((0, 0), (padd_left, padd_right)), mode='constant')
    elif w > h:
        padd_top = (w - h) // 2
        padd_bottom = w - h - padd_top
        image = np.pad(image, ((padd_top, padd_bottom), (0, 0)), mode='constant')

    h, w = image.shape[:2]
    if h != resolution or w != resolution:
        image = cv2.resize(image, resolution, interpolation=cv2.INTER_NEAREST if mode == 'nearest' else cv2.INTER_LINEAR)

    return image


def random_augment(x, y,):
    choice_flip = [True,False][np.random.randint(0, 2)]
    if choice_flip:
        x = np.flip(x, axis=0)
        y = np.flip(y, axis=0)

    choice_rot = [1, -1, 0][np.random.randint(0, 3)]
    x = np.rot90(x, k=choice_rot)
    y = np.rot90(y, k=choice_rot)
    return x, y


def seglabels_to_onehot(seglabels, num_classes=5):
    onehot = np.zeros(shape=(seglabels.shape[0], seglabels.shape[1], num_classes), dtype=np.int32)
    for c in range(num_classes):
        row, cols = np.where(seglabels == c)
        onehot[row, cols, int(c)] = 1
    return np.array(onehot)



In [ ]:
class TorchMRI3D_Dataloader(torch.utils.data.Dataset):
    def __init__(self, json_file, mode='train', num_classes=4, resolution=None, transforms=None, config={}, coords=None, skip_pixels=1.0):
        super(TorchMRI3D_Dataloader, self).__init__()
        assert num_classes == 4 or num_classes == 24 or num_classes == 35, "Only 4 or 24 classes are supported"
        self.json_file = json_file
        self.mode = mode
        self.coords=coords
        self.config = config
        self.resolution = resolution
        self.transforms = transforms
        self.num_classes = num_classes
        self.skip_pixels = skip_pixels
        self.build()

    def build(self):
        with open(self.json_file, 'r') as f:
            _data = json.load(f)

        self.data = _data[self.mode]
        if self.config.get('N_samples', None) is not None:
            self.data = self.data[:self.config['N_samples']]
        self.length = len(self.data)

    def __len__(self):
        return self.length

    def read_nib_file(self, file_path):
        img = nib.load(file_path).get_fdata()
        if img.shape[-1] == 1:
            img = img.squeeze(-1)
        return img

    def read_nib_volume(self, file_path):
        img = nib.load(file_path)
        return img.get_fdata()[:, 16:-16, 12:-12]

    def get_coords(self, h, w, d):
        xx = torch.linspace(-1, 1, h)
        yy = torch.linspace(-1, 1, w)
        zz = torch.linspace(-1, 1, d)

        coords = torch.meshgrid(xx, yy, zz)
        coords = torch.stack(coords, dim=-1)
        return coords


    def sample_from_3d(self, volume, segmentation):


        h,w,d = volume.shape

        coords_mtx = self.get_coords(h,w,d)
        volume_sampled = volume[::self.skip_pixels, ::self.skip_pixels, ::self.skip_pixels]
        segmentation_sampled = segmentation[::self.skip_pixels, ::self.skip_pixels, ::self.skip_pixels]
        coords_sampled = coords_mtx[::self.skip_pixels, ::self.skip_pixels, ::self.skip_pixels, ...]



        vs, ss, cs = volume_sampled.reshape(-1, 1), segmentation_sampled.reshape(-1, 1), coords_sampled.reshape(-1, 3)
        assert vs.shape[0] == ss.shape[0] == cs.shape[0], 'Shape mismatch for vs, ss, and cs'
        return vs, ss, cs


    def __getitem__(self, idx):
        data_dict = self.data[idx]
        vol_path = data_dict['img']
        seg_path = data_dict[f'seg{self.num_classes}']

        volume = self.read_nib_volume(vol_path)
        segmentation_integers = self.read_nib_volume(seg_path)
        coords = self.coords.clone()
        if self.skip_pixels != 1.0:
            volume, segmentation_integers, coords = self.sample_from_3d(volume, segmentation_integers)
        else:
            coords = self.get_coords(volume.shape[0], volume.shape[1], volume.shape[2]).reshape(-1, 3)
            volume = volume.reshape(-1, 1)
            segmentation_integers = segmentation_integers.reshape(-1, 1)


        seg_integer = segmentation_integers.copy()
        seg_onehot_labels = F.one_hot(torch.from_numpy(seg_integer).long(),
                                      num_classes=self.num_classes + 1).float()

        data_dict = {
            'img': torch.from_numpy(volume),
            'seg': seg_onehot_labels.squeeze(-2),
            'seg_integer': torch.from_numpy(seg_integer),
            'coords' : coords,
            'resolution': volume.shape,
        }
        return data_dict

class CLFFeature(torch.utils.data.Dataset):
    def __init__(self, path, mode):
        super(CLFFeature, self).__init__()
        self.path = path
        self.mode = mode

        self.all_files = glob.glob(osp.join(self.path, self.mode ,'*.pth'))
        self.length = len(self.all_files)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        st = time.time()
        data = torch.load(self.all_files[idx], weights_only=False)
        et = time.time()

        return data



class TorchMRI3D_DataloaderFast(torch.utils.data.Dataset):
    def __init__(self, json_file, mode='train', num_classes=4, resolution=None, transforms=None, config={}, coords=None, skip_pixels=1.0):
        super(TorchMRI3D_DataloaderFast, self).__init__()
        assert num_classes == 4 or num_classes == 24 or num_classes == 35, "Only 4 or 24 classes are supported"
        self.json_file = json_file
        self.mode = mode
        self.coords=coords
        self.config = config
        self.resolution = resolution
        self.transforms = transforms
        self.num_classes = num_classes
        self.skip_pixels = skip_pixels
        self.build()

    def build(self):
        with open(self.json_file, 'r') as f:
            _data = json.load(f)

        self.data = _data[self.mode]
        if self.config.get('N_samples', None) is not None:
            self.data = self.data[:self.config['N_samples']]
        self.length = len(self.data)

    def __len__(self):
        return self.length

    def read_nib_file(self, file_path):
        img = nib.load(file_path).get_fdata()
        if img.shape[-1] == 1:
            img = img.squeeze(-1)
        return img

    def read_nib_volume(self, file_path):
        img = nib.load(file_path)
        return img.get_fdata()[:, 16:-16, 12:-12]

    def get_coords(self, h, w, d):
        xx = torch.linspace(-1, 1, h)
        yy = torch.linspace(-1, 1, w)
        zz = torch.linspace(-1, 1, d)

        coords = torch.meshgrid(xx, yy, zz)
        coords = torch.stack(coords, dim=-1)
        return coords


    def sample_from_3d(self, volume, segmentation):


        h,w,d = volume.shape

        coords_mtx = self.get_coords(h,w,d)
        volume_sampled = volume[::self.skip_pixels, ::self.skip_pixels, ::self.skip_pixels]
        segmentation_sampled = segmentation[::self.skip_pixels, ::self.skip_pixels, ::self.skip_pixels]
        coords_sampled = coords_mtx[::self.skip_pixels, ::self.skip_pixels, ::self.skip_pixels, ...]



        return volume_sampled, segmentation_sampled, coords_sampled




    def __getitem__(self, idx):
        data_dict = self.data[idx]
        vol_path = data_dict['img']
        seg_path = data_dict[f'seg{self.num_classes}']

        volume = self.read_nib_volume(vol_path)
        segmentation_integers = self.read_nib_volume(seg_path)
        coords = self.coords.clone()
        if self.skip_pixels != 1.0:
            volume, segmentation_integers, coords = self.sample_from_3d(volume, segmentation_integers)
        else:
            coords = self.get_coords(volume.shape[0], volume.shape[1], volume.shape[2])
            volume = volume
            segmentation_integers = segmentation_integers


        seg_integer = segmentation_integers.copy()
        seg_onehot_labels = F.one_hot(torch.from_numpy(seg_integer).long(),

        data_dict = {
            'img': torch.from_numpy(volume),
            'seg': seg_onehot_labels,
            'seg_integer': torch.from_numpy(seg_integer),
            'coords' : coords,
            'resolution': volume.shape,
        }
        return data_dict


class TorchMRI3D_Dataloader_SR(torch.utils.data.Dataset):
    def __init__(self, json_file, mode='train', num_classes=4, resolution=None, transforms=None, config={}, coords=None, skip_pixels=1.0):
        super(TorchMRI3D_Dataloader_SR, self).__init__()
        assert num_classes == 4 or num_classes == 24 or num_classes == 35, "Only 4 or 24 classes are supported"
        self.json_file = json_file
        self.mode = mode
        self.coords=coords
        self.config = config
        self.resolution = resolution
        self.transforms = transforms
        self.num_classes = num_classes
        self.skip_pixels = skip_pixels
        self.build()

    def build(self):
        with open(self.json_file, 'r') as f:
            _data = json.load(f)

        self.data = _data[self.mode]
        if self.config.get('N_samples', None) is not None:
            self.data = self.data[:self.config['N_samples']]
        self.length = len(self.data)

    def __len__(self):
        return self.length

    def read_nib_file(self, file_path):
        img = nib.load(file_path).get_fdata()
        if img.shape[-1] == 1:
            img = img.squeeze(-1)
        return img

    def read_nib_volume(self, file_path):
        img = nib.load(file_path)
        return img.get_fdata()[:, 16:-16, 12:-12]

    def get_coords(self, h, w, d):
        xx = torch.linspace(-1, 1, h)
        yy = torch.linspace(-1, 1, w)
        zz = torch.linspace(-1, 1, d)

        coords = torch.meshgrid(xx, yy, zz)
        coords = torch.stack(coords, dim=-1)
        return coords


    def sample_from_3d(self, volume, segmentation):


        h,w,d = volume.shape

        coords_mtx = self.get_coords(h,w,d)
        volume_sampled = volume[::self.skip_pixels, ::self.skip_pixels, ::self.skip_pixels]
        segmentation_sampled = segmentation[::self.skip_pixels, ::self.skip_pixels, ::self.skip_pixels]
        coords_sampled = coords_mtx[::self.skip_pixels, ::self.skip_pixels, ::self.skip_pixels, ...]



        vs, ss, cs = volume_sampled.reshape(-1, 1), segmentation_sampled.reshape(-1, 1), coords_sampled.reshape(-1, 3)
        assert vs.shape[0] == ss.shape[0] == cs.shape[0], 'Shape mismatch for vs, ss, and cs'
        return vs, ss, cs





    def __getitem__(self, idx):
        data_dict = self.data[idx]
        vol_path = data_dict['img']
        seg_path = data_dict[f'seg{self.num_classes}']

        volume = self.read_nib_volume(vol_path)
        segmentation_integers = self.read_nib_volume(seg_path)
        coords = self.coords.clone()

        coords_hr = self.get_coords(volume.shape[0], volume.shape[1], volume.shape[2]).clone().reshape(-1, 3)
        volume_hr = volume.copy().reshape(-1, 1)
        segmentation_integers_hr = segmentation_integers.copy().reshape(-1, 1)
        if self.skip_pixels != 1.0:
            volume, segmentation_integers, coords = self.sample_from_3d(volume, segmentation_integers)
        else:
            coords = self.get_coords(volume.shape[0], volume.shape[1], volume.shape[2]).reshape(-1, 3)
            volume = volume.reshape(-1, 1)
            segmentation_integers = segmentation_integers.reshape(-1, 1)


        seg_integer = segmentation_integers.copy()
        seg_onehot_labels = F.one_hot(torch.from_numpy(seg_integer).long(),
                                      num_classes=self.num_classes + 1).float()

        seg_onehot_labels_hr = F.one_hot(torch.from_numpy(segmentation_integers_hr).long(),
                                      num_classes=self.num_classes + 1).float()

        data_dict = {
            'img': torch.from_numpy(volume),
            'seg': seg_onehot_labels.squeeze(-2),
            'seg_integer': torch.from_numpy(seg_integer),
            'coords' : coords,
            'resolution': volume.shape,
            'coords_hr' : coords_hr,
            'img_hr': torch.from_numpy(volume_hr),
            'seg_hr': seg_onehot_labels_hr.squeeze(-2),
            'seg_integer_hr': torch.from_numpy(segmentation_integers_hr),
        }
        return data_dict


## 6. Pipeline

In [ ]:
torch.manual_seed(422)
INNER_STEPS = 2
RANDOM_AUGMENT = False
TEST_RUN_STEPS = VAL_STEPS  = 100 # 2 #100#50
SKIP_PIXELS = 4
VAL_META_STEPS =  100
OUTER_LOOP_ITERATIONS =  3000 #300 #300  # 5000
NUM_CLASSES = 4
NUM_CLASSES_AND_ONE = 4 + 1

RES = (160,160,200)
VAL_RES = RES = [160//SKIP_PIXELS, 160//SKIP_PIXELS, 200//SKIP_PIXELS]

NORMALIZE_FEATURES = False

# GAMMA = 1.0 # 2.0
# HF = 128

nonlin = 'siren'
inr_config = {"in_features":3, "out_features": 1, "hidden_features": 256, "hidden_layers": 4, }
segmentation_config = {'hidden_features':[256,],#[128, 64],
                         'output_features' : NUM_CLASSES_AND_ONE}

In [ ]:
config_file = "/content/oasis_splits_3d.json"


files_data = json.load(open(config_file, 'r'))
train_files = files_data['train']
val_files = files_data['val']
test_files = files_data['test']

print(len(train_files), len(val_files), len(test_files))
set_train = set([x['img'] for x in train_files])
set_val = set([x['img'] for x in val_files])
set_test = set([x['img'] for x in test_files])

if len(set_train.intersection(set_val)) > 0 or  len(set_train.intersection(set_test)) > 0 or  len(set_val.intersection(set_test)) > 0:
    print("WARNING: OVERLAPPING DATA SPLITS")
else:
    print("No overlap in data splits. GOOD TO GO!!!!!!!!!!!")

214 100 100
No overlap in data splits. GOOD TO GO!!!!!!!!!!!


In [ ]:
inr_seg_model = SirenSegINR(
    inr_type='siren',
    inr_config=inr_config,
    segmentation_config=segmentation_config,
    normalize_features=NORMALIZE_FEATURES,

).float().cuda()

from torchinfo import summary
summary(inr_seg_model)

INR(
  (loss_function): MSELoss()
  (net): Sequential(
    (0): ResidualDualFreqSineLayer(
      (linear): Linear(in_features=3, out_features=256, bias=True)
      (gate): Linear(in_features=256, out_features=256, bias=True)
      (res_proj): Linear(in_features=3, out_features=256, bias=False)
    )
    (1): ResidualDualFreqSineLayer(
      (linear): Linear(in_features=256, out_features=256, bias=True)
      (gate): Linear(in_features=256, out_features=256, bias=True)
      (res_proj): Identity()
    )
    (2): ResidualDualFreqSineLayer(
      (linear): Linear(in_features=256, out_features=256, bias=True)
      (gate): Linear(in_features=256, out_features=256, bias=True)
      (res_proj): Identity()
    )
    (3): ResidualDualFreqSineLayer(
      (linear): Linear(in_features=256, out_features=256, bias=True)
      (gate): Linear(in_features=256, out_features=256, bias=True)
      (res_proj): Identity()
    )
    (4): ResidualDualFreqSineLayer(
      (linear): Linear(in_features=256, ou

Layer (type:depth-idx)                             Param #
SirenSegINR                                        --
├─INR: 1-1                                         --
│    └─MSELoss: 2-1                                --
│    └─Sequential: 2-2                             --
│    │    └─ResidualDualFreqSineLayer: 3-1         67,585
│    │    └─ResidualDualFreqSineLayer: 3-2         131,585
│    │    └─ResidualDualFreqSineLayer: 3-3         131,585
│    │    └─ResidualDualFreqSineLayer: 3-4         131,585
│    │    └─ResidualDualFreqSineLayer: 3-5         131,585
│    │    └─Linear: 3-6                            257
├─SegmentationModel: 1-2                           --
│    └─Sequential: 2-3                             --
│    │    └─Linear: 3-7                            65,792
│    │    └─LeakyReLU: 3-8                         --
│    │    └─Linear: 3-9                            1,285
Total params: 661,259
Trainable params: 661,259
Non-trainable params: 0

In [ ]:
meta_learner = INRMetaLearner(
    model=inr_seg_model,
    inner_steps=INNER_STEPS,
    config={'inner_lr':1e-4, 'outer_lr':1e-4},
    custom_loss_fn = LossFunction(
        {'mse_loss':MSELoss(alpha=1.0, reduction='weighted_mean', zero_weight=0.1),
         'focal_loss' : FocalSemanticLoss(alpha=1.0, gamma=3.0),
        },
    ),
     outer_optimizer='adam',
    inner_loop_loss_fn=None,
)


In [ ]:
coords_tmp = get_coords2d(RES[0], RES[1]).float().cuda()[None,...]
print(coords_tmp.shape)

torch.Size([1, 1600, 2])


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [ ]:
NUM_VAL_EXAMPLES = 100

In [ ]:
train_ds = TorchMRI3D_Dataloader(json_file=config_file, mode='train', resolution=RES, coords=coords_tmp, config={'augment':RANDOM_AUGMENT}, num_classes=NUM_CLASSES, skip_pixels=SKIP_PIXELS)
val_ds = TorchMRI3D_Dataloader(json_file=config_file, mode='val', resolution=RES, coords=coords_tmp, config={'augment':RANDOM_AUGMENT, 'N_samples':NUM_VAL_EXAMPLES}, num_classes=NUM_CLASSES, skip_pixels=SKIP_PIXELS)
test_ds = TorchMRI3D_Dataloader(json_file=config_file, mode='test', resolution=RES, coords=coords_tmp, config={'augment':RANDOM_AUGMENT}, num_classes=NUM_CLASSES, skip_pixels=SKIP_PIXELS)
print(len(train_ds), len(val_ds), len(test_ds))

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=1, shuffle=False)
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=1, shuffle=False)
test_dl = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=False)

214 100 100


In [ ]:
best_weights = deepcopy(meta_learner.model_params)
best_inr_weights = None
best_classifier_weights = None
val_dice_score = []
val_iou_scores = []
val_psnr_scores = []
best_val_psnr = 0
best_val_dice_score = 0


### 6.1. Train theta and phi

In [ ]:
for i in range(OUTER_LOOP_ITERATIONS//len(train_dl)):

    pbar = tqdm(enumerate(train_dl), total=len(train_dl))
    for ix, data in pbar:
        img = data['img'].float().cuda()
        seg = data['seg'].float().cuda()
        coords = data['coords'].float().cuda()
        seg_integer = data['seg_integer'].float().cuda()

        loss, loss_info = meta_learner.forward(coords, {'gt':img, 'seg':seg, 'seg_integer':seg_integer,'resolution' : data['resolution']})

        psnr = -10 * np.log10(loss_info.get('mse_loss',0.01))
        pbar.set_description(f"Loss: {loss.item():.5f} PSNR = {psnr.item():.5f} Dice={loss_info.get('dice_loss',-1):.4f} FL={loss_info.get('focal_loss', -1):.5f} TV={loss_info.get('tv_loss',-1):.5f}")
        pbar.refresh()

        if ix % VAL_META_STEPS == 0:

            val_dice_score = []
            val_iou_scores = []
            val_psnr_scores = []


            for val_ix, val in tqdm(enumerate(val_dl), total=len(val_dl), position=1):
                  val_img = val['img'].float().cuda()
                  val_seg = val['seg'].float().cuda()
                  val_coords = val['coords'].float().cuda()
                  val_seg_integer = val['seg_integer'].float().cuda()

                  render = meta_learner.render_inner_loop(val_coords, val_img, inner_loop_steps=VAL_STEPS)
                  segmentation_output = render['output']['segmentation_output'].detach()
                  segmentation_output = nn.functional.softmax(segmentation_output, dim=-1)
                  segmentation_output = segmentation_output.argmax(dim=-1).detach().reshape(VAL_RES).cpu().numpy()
                  img_recon = render['output']['inr_output'][0].reshape(VAL_RES).detach().cpu().numpy()
                  segmentation_output_onehot = torch.nn.functional.one_hot(torch.tensor(segmentation_output), num_classes=NUM_CLASSES_AND_ONE)
                  val_reshaped = val_img[0].detach().cpu().numpy().reshape(VAL_RES)
                  val_seg_reshaped = val_seg_integer.detach().cpu().numpy().reshape(VAL_RES)

                  mse_val = img_recon[...,40:80].flatten() - val_reshaped[...,40:80].flatten()
                  mse_val = np.mean(mse_val**2)
                  psnr = psnr = -10 * np.log10(mse_val)
                  val_psnr_scores.append(float(psnr))
                  dice_score = multiclass_dice_score_3d(segmentation_output_onehot.cuda(), val_seg.reshape(VAL_RES[0], VAL_RES[1], VAL_RES[2], -1).cuda(), num_classes=NUM_CLASSES_AND_ONE)
                  val_dice_score.append(float(dice_score.item()))


            if np.mean(val_dice_score) > best_val_dice_score:
                  best_val_psnr = np.mean(val_psnr_scores)
                  best_val_dice_score = np.mean(val_dice_score)
                  best_weights = deepcopy(meta_learner.model_params)
                  best_inr_weights = deepcopy(meta_learner.get_inr_parameters())
                  best_classifier_weights = deepcopy(meta_learner.get_segmentation_parameters())
                  best_idx= ix
                  print('updated dice score to ', best_val_dice_score)
                  torch.save({'inr_seg_model':inr_seg_model.state_dict(),
                              'best_inr_weights':best_inr_weights,
                              'best_classifier_weights':best_classifier_weights},
                              f"/content/drive/MyDrive/dumps/weights3d_num_classes_{NUM_CLASSES}_IS_{INNER_STEPS}.pth")

            print(f"Mean PSNR={np.mean(val_psnr_scores):.5f} +/- {np.std(val_psnr_scores):.5f}")

            print(f"Mean Dice={np.mean(val_dice_score):.5f} +/- {np.std(val_dice_score):.5f}")


print("Best weights from Dice=", best_val_dice_score)

  0%|          | 0/214 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

updated dice score to  0.0831591820716858
Mean PSNR=38.66787 +/- 0.46732
Mean Dice=0.08316 +/- 0.00372


  0%|          | 0/100 [00:00<?, ?it/s]

updated dice score to  0.7104143446683884
Mean PSNR=41.80581 +/- 0.41821
Mean Dice=0.71041 +/- 0.01556


...

Mean PSNR=57.98310 +/- 4.84803
Mean Dice=0.06146 +/- 0.01176


  0%|          | 0/100 [00:00<?, ?it/s]

Mean PSNR=56.63256 +/- 5.88160
Mean Dice=0.06222 +/- 0.01185


  0%|          | 0/100 [00:00<?, ?it/s]

Mean PSNR=56.97448 +/- 5.20675
Mean Dice=0.06195 +/- 0.01183
Best weights from Dice= 0.731304578781128


In [ ]:
torch.save({'inr_seg_model':inr_seg_model.state_dict(),'best_inr_weights':best_inr_weights,
                'best_classifier_weights':best_classifier_weights}, f"/content/drive/MyDrive/dumpsweights3d_num_classes_{NUM_CLASSES}_IS_{INNER_STEPS}.pth")

### 6.2. Finetune MetaSeg's segmentation head



In [ ]:
SAVE_PATHS = "/content/drive/MyDrive/intermediate_vectors"

In [ ]:
os.makedirs(os.path.join(SAVE_PATHS, "train"), exist_ok=True)
os.makedirs(os.path.join(SAVE_PATHS, "val"), exist_ok=True)

In [ ]:
weights_file = "/content/drive/MyDrive/dumps/weights3d_num_classes_4_IS_2.pth"

In [ ]:
weights_from_metalearning = torch.load(weights_file, map_location=torch.device('cpu'), weights_only=False)
inr_seg_model_wts = weights_from_metalearning['inr_seg_model']
best_inr_weights = weights_from_metalearning['best_inr_weights']
best_classifier_weights = weights_from_metalearning['best_classifier_weights']


In [ ]:
train_clf_features = []
val_clf_features = []

pbar_train = tqdm(enumerate(train_dl), total=len(train_dl), position=0)
for train_ix, train_data in pbar_train:
  tmp_save_check =  osp.join(SAVE_PATHS,"train" , f"train_{train_ix}.pth")
  if osp.isfile(tmp_save_check):
    continue

  inr_model_train = INR(**inr_config).float().cuda()
  inr_model_train.load_state_dict({k.replace("inr.",""): v.clone().detach() for k,v in deepcopy(best_inr_weights).items()}, strict=False)
  inr_model_train.compile()

  train_img = train_data['img'].float().cuda()
  train_seg = train_data['seg'].float().cuda()
  train_coords = train_data['coords'].float().cuda()

  inr_model_train.fit(train_coords, train_img, epochs=TEST_RUN_STEPS, disable_tqdm=True)
  train_img_output, train_img_features = inr_model_train.forward_w_features(train_coords)

  data_dt = {
    'seg': train_seg.detach().clone().cpu().numpy(),
    'img': train_img_output.detach().clone().cpu().numpy(),
    'features': train_img_features[-2].detach().clone().cpu().numpy()
  }

  os.makedirs(osp.join(SAVE_PATHS,"train"), exist_ok=True)
  torch.save(data_dt, osp.join(SAVE_PATHS,"train" , f"train_{train_ix}.pth"),pickle_protocol=0)


  0%|          | 0/214 [00:00<?, ?it/s]

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-1918/2680988452.py", line 5, in <cell line: 0>
    for train_ix, train_data in pbar_train:
                                ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tqdm/notebook.py", line 250, in __iter__
    yield from it
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
               ^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 801, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local


KeyboardInterrupt



In [ ]:
pbar_val = tqdm(enumerate(val_dl), total=len(val_dl), position=0)
for val_ix, val_data in pbar_val:
  _tmp_save_check =  osp.join(SAVE_PATHS,"val" , f"val_{train_ix}.pth")
  if osp.isfile(_tmp_save_check):
     continue

  inr_model_val = INR(**inr_config).float().cuda()
  inr_model_val.load_state_dict({k.replace("inr.",""): v.clone().detach() for k,v in deepcopy(best_inr_weights).items()}, strict=False)
  inr_model_val.compile()

  val_img = val_data['img'].float().cuda()
  val_seg = val_data['seg'].float().cuda()
  val_coords = val_data['coords'].float().cuda()

  inr_model_val.fit(val_coords, val_img, epochs=TEST_RUN_STEPS, disable_tqdm=True)
  val_img_output, val_img_features = inr_model_val.forward_w_features(val_coords)

  val_dt = {
    'seg':val_seg.detach().clone().cpu().numpy(),
    'img': val_img.detach().clone().cpu().numpy(),
    'features': val_img_features[-2].detach().clone().cpu().numpy(),
  }


  os.makedirs(osp.join(SAVE_PATHS, "val"), exist_ok=True)
  torch.save(val_dt, osp.join(SAVE_PATHS,"val" , f"val_{val_ix}.pth"), pickle_protocol=0)

In [ ]:
test_clf_features = True
test_features = []
if test_clf_features:
  pbar_test = tqdm(enumerate(test_dl), total=len(test_dl), position=0)
  for test_ix, test_data in pbar_test:
     inr_model_test = INR(**inr_config).float().cuda()
     inr_model_test.load_state_dict({k.replace("inr.",""): v.clone().detach() for k,v in deepcopy(best_inr_weights).items()}, strict=False)
     inr_model_test.compile()

     test_img = test_data['img'].float().cuda()
     test_seg = test_data['seg'].float().cuda()
     test_coords = test_data['coords'].float().cuda()

     inr_model_test.fit(test_coords, test_img, epochs=TEST_RUN_STEPS, disable_tqdm=True)
     test_img_output, test_img_features = inr_model_test.forward_w_features(test_coords)

     test_dt = {
      'seg':test_seg.detach().clone().cpu().numpy(),
      'img': test_img.detach().clone().cpu().numpy(),

       'features': test_img_features[-2].detach().clone().cpu().numpy(),
     }


     os.makedirs(osp.join(SAVE_PATHS,"test"), exist_ok=True)
     torch.save(test_dt, osp.join(SAVE_PATHS,"test" , f"test_{test_ix}.pth"), pickle_protocol=0)

  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
data_dir_feature_vecs = SAVE_PATHS

In [ ]:
train_clf_features_ds = CLFFeature(data_dir_feature_vecs, mode='train')
val_clf_ds = CLFFeature(data_dir_feature_vecs, mode='val')
test_clf_ds = CLFFeature(data_dir_feature_vecs, mode='test')

train_clf_dl = torch.utils.data.DataLoader(train_clf_features_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
val_clf_dl = torch.utils.data.DataLoader(val_clf_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
test_clf_dl = torch.utils.data.DataLoader(test_clf_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
CLASSIFIER_FINETUNE_EPOCHS = 100

In [ ]:

classifier_model = deepcopy(inr_seg_model.segmentation_head)

classifier_weights = deepcopy(best_classifier_weights)
try:
    classifier_model.load_state_dict(classifier_weights['final_clf_weights'])
except:
    classifier_weights = deepcopy({k.replace("segmentation_head.segmentation_head","segmentation_head"):v.clone().detach() for k,v in best_classifier_weights.items()})


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/mdd/train"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_classifier_progress(recon_logits, gt_seg_tensor, gt_real_tensor, epoch, train_id):

    pred_seg_array = torch.argmax(recon_logits.squeeze(), dim=-1).reshape(VAL_RES).detach().cpu().numpy()

    gt_seg_array = torch.argmax(gt_seg_tensor.reshape(VAL_RES[0], VAL_RES[1], VAL_RES[2], -1), dim=-1).detach().cpu().numpy()

    gt_real_array = gt_real_tensor.reshape(VAL_RES).detach().cpu().numpy()

    def write_mhd(data, name, is_label=False):
        img = sitk.GetImageFromArray(data.astype(np.uint16 if is_label else np.float32))
        sitk.WriteImage(img, f"{SAVE_DIR}/epoch_{epoch}_id_{train_id}_{name}.mhd")

    write_mhd(gt_real_array, "gt_real")
    write_mhd(gt_seg_array, "gt_seg", is_label=True)
    write_mhd(pred_seg_array, "inr_seg", is_label=True)

In [ ]:
LEARNING_RATE = 5e-5
FOCAL_LOSS_GAMMA = 3.0
ZERO_WT = 0.1

EXPERIMENT_NAME  = f"gamma_{FOCAL_LOSS_GAMMA}_INR_300it_skip_pixels_{SKIP_PIXELS}_continue"

classifier_opt = torch.optim.Adam(classifier_model.parameters(), lr=LEARNING_RATE)

print(classifier_model, list(classifier_weights.keys()), list(classifier_model.state_dict().keys()))

finetune_classifier_loss_fn = LossFunction({'focal_loss':FocalSemanticLoss(gamma=FOCAL_LOSS_GAMMA)})

final_classifier_weights = deepcopy(classifier_model.state_dict())
current_best_weights = {k: v.cpu().clone() for k, v in classifier_model.state_dict().items()}
best_val_score = float('inf')

pbar_epochs = tqdm(range(CLASSIFIER_FINETUNE_EPOCHS), position=0)
for epoch in pbar_epochs:
  print("Epoch:",epoch)

  avg_loss_per_set = 0.0

  for train_ix, train_data in enumerate(train_clf_dl):
    if (train_ix%10==0):
      print("Train id:",train_ix)
    train_seg = train_data['seg'].float().cuda()
    classifier_input = train_data['features'].float().cuda().squeeze(0).squeeze(0)
    if NORMALIZE_FEATURES:
      classifier_input = nn.functional.normalize(classifier_input, dim=-1)

    classifier_opt.zero_grad()
    classifier_output = classifier_model(classifier_input)
    classifier_output = classifier_output.unsqueeze(0).unsqueeze(0)
    loss, loss_info = finetune_classifier_loss_fn({'output':{'segmentation_output':classifier_output}, 'seg':train_seg})
    loss.backward()
    classifier_opt.step()

    avg_loss_per_set += float(loss.item())

    if epoch % 10 == 0 and train_ix < 10:
            save_classifier_progress(
                recon_logits=classifier_output,
                gt_seg_tensor=train_seg,
                gt_real_tensor=train_data['img'].float().cuda(),
                epoch=epoch,
                train_id=train_ix
            )

  avg_loss_per_set /= len(train_clf_dl)
  pbar_epochs.set_description(f"Loss (clf): {avg_loss_per_set:.5f}. Best Val Loss(clf): {best_val_score:.5f}")
  pbar_epochs.refresh()

  if epoch % VAL_META_STEPS == 0:
    with torch.no_grad():
      avg_val_loss = 0.0
      for val_ix, val_data in enumerate(val_clf_dl):
        val_seg = val_data['seg'].float().cuda()
        classifier_val_input = val_data['features'].float().cuda().squeeze(0).squeeze(0)
        if NORMALIZE_FEATURES:
          classifier_val_input = nn.functional.normalize(classifier_val_input, dim=-1)

        classifier_val_output = classifier_model(classifier_val_input)
        classifier_val_output = classifier_val_output.unsqueeze(0).unsqueeze(0)
        val_loss, val_loss_info = finetune_classifier_loss_fn({'output':{'segmentation_output':classifier_val_output}, 'seg':val_seg})

        avg_val_loss += float(val_loss.item())

      avg_val_loss = avg_val_loss / len(val_clf_dl)
      pbar_epochs.set_description(f"Loss (clf): {avg_loss_per_set:.5f} Val Loss (clf): {avg_val_loss:.5f}")
      pbar_epochs.refresh()

      print(f"Epoch {epoch}: Val Loss = {avg_val_loss:.6f} | Best So Far = {best_val_score:.6f}")
      if avg_val_loss < best_val_score:
        tqdm.write(f"New Best! Saving weights...")
        best_val_score = avg_val_loss
        final_classifier_weights = deepcopy(classifier_model.state_dict())
        current_best_weights = {k: v.cpu().clone() for k, v in classifier_model.state_dict().items()}
        tqdm.write(f'updated best val score to {best_val_score}')
        torch.save({'final_clf_weights':current_best_weights, 'focal_loss_gamma':FOCAL_LOSS_GAMMA, 'zero_wt':ZERO_WT},
                    f"/content/drive/MyDrive/dumps/weights_3dclassifierfinal_final_weights_LR_{LEARNING_RATE}_exp_{EXPERIMENT_NAME}.pth")
      else:
        tqdm.write(f"No improvement.")

SegmentationModel(
  (segmentation_head): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=256, out_features=5, bias=True)
  )
) ['segmentation_head.0.weight', 'segmentation_head.0.bias', 'segmentation_head.2.weight', 'segmentation_head.2.bias'] ['segmentation_head.0.weight', 'segmentation_head.0.bias', 'segmentation_head.2.weight', 'segmentation_head.2.bias']


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch: 0
Train id: 0
Train id: 10
Train id: 20
Train id: 30
Train id: 40
Train id: 50
Train id: 60
Train id: 70
Train id: 80
Train id: 90
Train id: 100
Train id: 110
Train id: 120
Train id: 130
Train id: 140
Train id: 150
Train id: 160
Train id: 170
Train id: 180
Train id: 190
Train id: 200
Train id: 210
Epoch 0: Val Loss = 0.028924 | Best So Far = inf
New Best! Saving weights...
updated best val score to 0.028923789374530315
Epoch: 1
Train id: 0
Train id: 10
Train id: 20
Train id: 30
Train id: 40
Train id: 50
Train id: 60
Train id: 70
Train id: 80
Train id: 90
Train id: 100
Train id: 110
Train id: 120
Train id: 130
Train id: 140
Train id: 150
Train id: 160
Train id: 170
Train id: 180
Train id: 190
Train id: 200
Train id: 210
Epoch: 2
Train id: 0
Train id: 10
Train id: 20
Train id: 30
Train id: 40
Train id: 50
Train id: 60
Train id: 70
Train id: 80
Train id: 90
Train id: 100
Train id: 110
Train id: 120
Train id: 130
Train id: 140
Train id: 150
Train id: 160
Train id: 170
Train id: 180


In [ ]:
save_path = f"/content/drive/MyDrive/dumps/weights_3dclassifier_best_final_weights_LR_{LEARNING_RATE}_{EXPERIMENT_NAME}.pth"
torch.save({
    'final_clf_weights': current_best_weights,
    'focal_loss_gamma': FOCAL_LOSS_GAMMA,
    'zero_wt': ZERO_WT
}, save_path)

In [ ]:
epoch = 100
checkpoint = {
    'epoch': epoch,
    'model_state_dict': classifier_model.state_dict(),
    'optimizer_state_dict': classifier_opt.state_dict(),
    'best_val_score': best_val_score,
    'experiment_name': EXPERIMENT_NAME
}

checkpoint_path = f"/content/drive/MyDrive/dumps/checkpoint_epoch_{epoch}.pth"
torch.save(checkpoint, checkpoint_path)

In [ ]:
checkpoint_path = f"/content/drive/MyDrive/dumps/checkpoint_epoch_100.pth"

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path)

    classifier_model.load_state_dict(checkpoint['model_state_dict'])
    classifier_opt.load_state_dict(checkpoint['optimizer_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    best_val_score = checkpoint['best_val_score']
    EXPERIMENT_NAME = checkpoint['experiment_name']

    print(f"Resuming from Epoch {start_epoch} with Best Val Loss: {best_val_score:.6f}")
else:
    print("No checkpoint found. Starting from scratch.")
    start_epoch = 0
    best_val_score = float('inf')

Loading checkpoint from /content/drive/MyDrive/dumps/checkpoint_epoch_100.pth...
Resuming from Epoch 101 with Best Val Loss: 0.028924


In [ ]:
classifier_model = classifier_model.cuda().train()
CLASSIFIER_FINETUNE_EPOCHS = 200

pbar_epochs = tqdm(range(start_epoch, CLASSIFIER_FINETUNE_EPOCHS), desc="Total Epochs", position=0)

for epoch in pbar_epochs:
    print("Epoch:",epoch)
    avg_loss_per_set = 0.0
    classifier_model.train()

    for train_ix, train_data in enumerate(train_clf_dl):
        if (train_ix%10==0):
          print("Train id:",train_ix)

        train_seg = train_data['seg'].float().cuda()
        classifier_input = train_data['features'].float().cuda().squeeze(0).squeeze(0)

        if NORMALIZE_FEATURES:
            classifier_input = torch.nn.functional.normalize(classifier_input, dim=-1)

        classifier_opt.zero_grad()
        classifier_output = classifier_model(classifier_input)
        classifier_output = classifier_output.unsqueeze(0).unsqueeze(0)

        loss, loss_info = finetune_classifier_loss_fn(
            {'output': {'segmentation_output': classifier_output}, 'seg': train_seg}
        )

        loss.backward()
        classifier_opt.step()

        avg_loss_per_set += float(loss.item())

    avg_loss_per_set /= len(train_clf_dl)
    pbar_epochs.set_description(f"Loss (clf): {avg_loss_per_set:.5f}. Best Val Loss(clf): {best_val_score:.5f}")
    pbar_epochs.refresh()

    if epoch % VAL_META_STEPS == 0:
        classifier_model.eval()
        avg_val_loss = 0.0

        with torch.no_grad():
            for val_ix, val_data in enumerate(val_clf_dl):
                val_seg = val_data['seg'].float().cuda()
                classifier_val_input = val_data['features'].float().cuda().squeeze(0).squeeze(0)

                if NORMALIZE_FEATURES:
                    classifier_val_input = torch.nn.functional.normalize(classifier_val_input, dim=-1)

                classifier_val_output = classifier_model(classifier_val_input)
                classifier_val_output = classifier_val_output.unsqueeze(0).unsqueeze(0)

                val_loss, _ = finetune_classifier_loss_fn(
                    {'output': {'segmentation_output': classifier_val_output}, 'seg': val_seg}
                )
                avg_val_loss += float(val_loss.item())

            avg_val_loss /= len(val_clf_dl)

            pbar_epochs.set_description(f"Loss (clf): {avg_loss_per_set:.5f} Val Loss (clf): {avg_val_loss:.5f}")
            pbar_epochs.refresh()

            if avg_val_loss < best_val_score:
                tqdm.write(f"New Best! Saving weights...")
                best_val_score = avg_val_loss
                final_classifier_weights = deepcopy(classifier_model.state_dict())
                current_best_weights = {k: v.cpu().clone() for k, v in classifier_model.state_dict().items()}
                tqdm.write(f'updated best val score to {best_val_score}')
                torch.save({'final_clf_weights':current_best_weights, 'focal_loss_gamma':FOCAL_LOSS_GAMMA, 'zero_wt':ZERO_WT},
                            f"/content/drive/MyDrive/dumps/weights_3dclassifierfinal_final_weights_LR_{LEARNING_RATE}_exp_{EXPERIMENT_NAME}.pth")
            else:
                tqdm.write(f"Epoch {epoch}: No improvement.")

print("Finished training session.")

Total Epochs:   0%|          | 0/99 [00:00<?, ?it/s]

Epoch: 101
Train id: 0
Train id: 10
Train id: 20
Train id: 30
Train id: 40
Train id: 50
Train id: 60
Train id: 70
Train id: 80
Train id: 90
Train id: 100
Train id: 110
Train id: 120
Train id: 130
Train id: 140
Train id: 150
Train id: 160
Train id: 170
Train id: 180
Train id: 190
Train id: 200
Train id: 210
Epoch: 102
Train id: 0
Train id: 10
Train id: 20
Train id: 30


KeyboardInterrupt: 

### 6.3. Test-time. Fit Pixels, Get Labels!

In [ ]:
SAVE_PATHS = "/content/drive/MyDrive/intermediate_vectors"
data_dir_feature_vecs = SAVE_PATHS

test_clf_ds = CLFFeature(data_dir_feature_vecs, mode='test')

test_clf_dl = torch.utils.data.DataLoader(test_clf_ds, batch_size=1, shuffle=False, num_workers=1, pin_memory=False)

print(len(test_clf_dl))
print(len(test_clf_ds.all_files))

100
100


In [ ]:
segmentation_clf_weights = "/content/drive/MyDrive/dumps/weights_3dclassifier_best_final_weights_LR_5e-05_gamma_3.0_INR_300it_skip_pixels_4_continue.pth"

weights_saved = torch.load(segmentation_clf_weights)


final_clf_weights = weights_saved['final_clf_weights']

In [ ]:
classifier_model = deepcopy(inr_seg_model.segmentation_head)
classifier_model.load_state_dict(final_clf_weights)
classifier_model.eval()

SegmentationModel(
  (segmentation_head): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=256, out_features=5, bias=True)
  )
)

In [ ]:
def plot_mri_planes(pred_vol, gt_vol, dice_val, sample_idx):

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), facecolor='white')

    mid_h, mid_w, mid_d = [s // 2 for s in pred_vol.shape]


    pred_slices = [pred_vol[:, :, mid_d], pred_vol[mid_h, :, :], pred_vol[:, mid_w, :]]
    gt_slices = [gt_vol[:, :, mid_d], gt_vol[mid_h, :, :], gt_vol[:, mid_w, :]]
    titles = ['Axial', 'Coronal', 'Sagittal']

    for i in range(3):

        axes[0, i].imshow(pred_slices[i], cmap='nipy_spectral', interpolation='nearest')
        axes[0, i].set_title(f"MetaSeg {titles[i]} (Dice: {dice_val:.3f})")
        axes[0, i].axis('off')

        axes[1, i].imshow(gt_slices[i], cmap='nipy_spectral', interpolation='nearest')
        axes[1, i].set_title(f"GT {titles[i]}")
        axes[1, i].axis('off')

    plt.suptitle(f"Sample {sample_idx} Cross-Sections", fontsize=16)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

def plot_3d_segmentation(volume, sample_idx):

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    step = 2
    v_sub = volume[::step, ::step, ::step]

    z, y, x = np.where(v_sub > 0)
    colors = v_sub[v_sub > 0]

    scatter = ax.scatter(x, y, z, c=colors, cmap='nipy_spectral', s=1, alpha=0.6)

    ax.set_title(f"3D Reconstruction - Sample {sample_idx}")
    ax.view_init(elev=20, azim=45)
    plt.show()
    plt.close(fig)

In [ ]:
def plot_polished_mri_planes(pred_vol, gt_vol, dice_val, sample_idx):
    """
    Polished multi-planar view with high-contrast, professional formatting.
    """
    plt.style.use('dark_background')
    fig, axes = plt.subplots(2, 3, figsize=(15, 9), facecolor='black')

    mid = [s // 2 for s in pred_vol.shape]


    slices_pred = [pred_vol[:, :, mid[2]], pred_vol[mid[0], :, :], pred_vol[:, mid[1], :]]
    slices_gt = [gt_vol[:, :, mid[2]], gt_vol[mid[0], :, :], gt_vol[:, mid[1], :]]
    titles = ['Axial Plane', 'Coronal Plane', 'Sagittal Plane']

    colors_list = ['black', 'red', '#6699CC', 'purple', 'white']
    cmap_labels = ListedColormap(colors_list)

    for i in range(3):

        axes[0, i].imshow(slices_pred[i], cmap=cmap_labels, interpolation='nearest')
        axes[0, i].set_title(f"MetaSeg {titles[i]}\nDice: {dice_val:.3f}", color='white', fontsize=12)

        axes[1, i].imshow(slices_gt[i], cmap=cmap_labels, interpolation='nearest')
        axes[1, i].set_title(f"GT {titles[i]}", color='white', fontsize=12)

        for row in range(2):
            axes[row, i].axis('off')

    plt.suptitle(f"Sample {sample_idx} Cross-Sections", fontsize=16, color='white', y=1.02)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

def plot_layered_3d_segmentation(volume, sample_idx):

    fig = plt.figure(figsize=(16, 8), facecolor='white')


    label_colors = {
        1: (1.0, 0.0, 0.0, 0.8), # Red
        2: (0.4, 0.6, 0.8, 0.8), # Blue
        3: (0.5, 0.0, 0.5, 0.9), # Purple
    }

    step = 1 if volume.shape[0] < 128 else 2
    v_sub = volume[::step, ::step, ::step]

    ax1 = fig.add_subplot(121, projection='3d')
    ax2 = fig.add_subplot(122, projection='3d')

    for label_id, color in label_colors.items():
        mask = (v_sub == label_id)
        if np.any(mask):
            coords = np.where(mask)
            x, y, z = coords

            for ax in [ax1, ax2]:
                ax.scatter(z, y, x, color=color, s=1, marker='s', edgecolors='none')

    ax1.view_init(elev=20, azim=110)
    ax1.set_title("3D Sagittal Plane View", fontsize=14)

    ax2.view_init(elev=50, azim=30)
    ax2.set_title("3D Axial Plane View", fontsize=14)

    for ax in [ax1, ax2]:
        ax.axis('off')
        ax.set_facecolor('white')

    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:


def plot_mri_3d_comparison(gt_img, recon_img, gt_seg, pred_seg, num_classes=5):
    fig = plt.figure(figsize=(20, 10))

    def setup_3d_ax(ax, img, seg, title):

        step = 2
        z, y, x = (img[::step, ::step, ::step] > 0.1).nonzero()
        vals = img[::step, ::step, ::step][img[::step, ::step, ::step] > 0.1]
        ax.scatter(x*step, y*step, z*step, c=vals, cmap='gray', alpha=0.05, s=1)

        colors = plt.cm.get_cmap('tab10', num_classes)
        for c in range(1, num_classes):
            mask = (seg == c)
            if mask.any():
                zs, ys, xs = mask.nonzero()
                ax.scatter(xs[::step], ys[::step], zs[::step],
                           color=colors(c), label=f'Class {c}', s=3, alpha=0.6)

        ax.set_title(title)
        ax.axis('off')

    ax1 = fig.add_subplot(121, projection='3d')
    setup_3d_ax(ax1, gt_img, gt_seg, "GROUND TRUTH (MRI + Seg)")

    ax2 = fig.add_subplot(122, projection='3d')
    setup_3d_ax(ax2, recon_img, pred_seg, "MODEL PREDICTION (INR + Seg)")

    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

In [ ]:
!pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 50.6 MB/s eta 0:00:00


In [ ]:


def export_with_itk_to_drive(recon_3d, gt_img, pred_seg, gt_seg, index, spacing=(1.0, 1.0, 1.0)):

    save_dir = "/content/drive/MyDrive/mdd/test"

    if not os.path.exists(save_dir):
        os.makedirs(save_dir, exist_ok=True)

    error_map = np.abs(gt_img - recon_3d).astype(np.float32)

    def to_itk(array, is_label=False):
        img_itk = sitk.GetImageFromArray(array.transpose(2, 0, 1))
        img_itk.SetSpacing(spacing)
        if is_label:
            return sitk.Cast(img_itk, sitk.sitkUInt16)
        return sitk.Cast(img_itk, sitk.sitkFloat32)

    itk_recon = to_itk(recon_3d)
    itk_gt    = to_itk(gt_img)
    itk_error = to_itk(error_map)
    itk_pred_seg = to_itk(pred_seg, is_label=True)
    itk_gt_seg   = to_itk(gt_seg, is_label=True)

    sitk.WriteImage(itk_recon, os.path.join(save_dir, f"case_{index}_INR_Recon.mhd"))
    sitk.WriteImage(itk_gt, os.path.join(save_dir, f"case_{index}_GT_Real.mhd"))
    sitk.WriteImage(itk_error, os.path.join(save_dir, f"case_{index}_ErrorMap.mhd"))
    sitk.WriteImage(itk_pred_seg, os.path.join(save_dir, f"case_{index}_PredSeg.mhd"))
    sitk.WriteImage(itk_gt_seg, os.path.join(save_dir, f"case_{index}_GTSeg.mhd"))

    print(f"✅ ITK Export for case {index} saved to Drive: {save_dir}")

In [ ]:
inr_initialization_weights = deepcopy(best_inr_weights)
classifier_model.load_state_dict(final_classifier_weights)
classifier_model = classifier_model.cuda().eval()


dice_scores = []
psnr_scores = []

TEST_RUN_STEPS = 100
INR_FIT_LR = 1e-4
NUM_CLASSES = 5

pbar_test = tqdm(enumerate(test_dl), total=len(test_dl))

for test_ix, test_data in pbar_test:
    test_coords = test_data['coords'].float().cuda(non_blocking=True)
    test_img    = test_data['img'].float().cuda(non_blocking=True)
    test_seg    = test_data['seg'].float().cuda(non_blocking=True)

    test_inr_render_model = INR(**inr_config).float().cuda()

    model_sd = test_inr_render_model.state_dict()
    load_sd = {}
    for k, v in inr_initialization_weights.items():
        kk = k.replace("inr.", "")
        if kk in model_sd:
            load_sd[kk] = v.detach().clone()
    test_inr_render_model.load_state_dict(load_sd, strict=False)

    test_inr_render_model.optimizer = torch.optim.Adam(
        test_inr_render_model.parameters(), lr=INR_FIT_LR
    )
    test_inr_render_model.loss_function = nn.MSELoss()
    test_inr_render_model.scheduler = torch.optim.lr_scheduler.LambdaLR(
        test_inr_render_model.optimizer, lr_lambda=lambda epoch: 1.0
    )

    test_inr_render_model.fit(
        test_coords, test_img, epochs=TEST_RUN_STEPS, disable_tqdm=True
    )

    with torch.no_grad():
        test_img_output, test_img_features = test_inr_render_model.forward_w_features(test_coords)

        classifier_input = test_img_features[-2]
        if NORMALIZE_FEATURES:
            classifier_input = nn.functional.normalize(classifier_input, dim=-1)

        classifier_output = classifier_model(classifier_input)


        logits = classifier_output.reshape(VAL_RES[0], VAL_RES[1], VAL_RES[2], NUM_CLASSES)

        seg_pred = torch.argmax(torch.softmax(logits, dim=-1), dim=-1)

        seg_pred_onehot = torch.nn.functional.one_hot(seg_pred, num_classes=NUM_CLASSES)

    cur_psnr = psnr2(
        test_img.reshape(-1).detach().cpu().numpy(),
        test_img_output.reshape(-1).detach().cpu().numpy()
    )


    test_seg_reshaped = test_seg.reshape(VAL_RES[0], VAL_RES[1], VAL_RES[2], -1)
    cur_dice = multiclass_dice_score_3d(
        seg_pred_onehot.cuda(),
        test_seg_reshaped.cuda(),
        num_classes=NUM_CLASSES
    )

    psnr_scores.append(float(cur_psnr))
    dice_scores.append(float(cur_dice.item()))


    recon_np = test_img_output.view(VAL_RES).detach().cpu().numpy()
    gt_img_np = test_img.view(VAL_RES).detach().cpu().numpy()
    pred_seg_np = seg_pred.detach().cpu().numpy()
    gt_seg_np = torch.argmax(test_seg_reshaped, dim=-1).detach().cpu().numpy()

    export_with_itk_to_drive(recon_np, gt_img_np, pred_seg_np, gt_seg_np, test_ix)


    pbar_test.set_description(
        f"Test {test_ix+1}: PSNR={cur_psnr:.4f} Dice={float(cur_dice.item()):.4f}"
    )

    del test_inr_render_model, test_img_output, test_img_features, classifier_output, logits, seg_pred
    torch.cuda.empty_cache()


print("\n" + "---"*10 + " FINAL RESULTS " + "---"*10)
print(f"Mean Reconstruction PSNR: {np.mean(psnr_scores):.4f} +/- {np.std(psnr_scores):.4f}")
print(f"Mean Segmentation Dice:   {np.mean(dice_scores):.4f} +/- {np.std(dice_scores):.4f}")

  0%|          | 0/100 [00:00<?, ?it/s]

✅ ITK Export for case 0 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 1 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 2 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 3 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 4 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 5 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 6 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 7 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 8 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 9 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 10 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 11 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 12 saved to Drive: /content/drive/MyDrive/mdd/test
✅ ITK Export for case 13 saved to Drive: /content/drive/MyDri

In [ ]:
print("Average Segmentation Dice = ", np.mean(dice_scores), "+/-", np.std(dice_scores))

Average Segmentation Dice =  0.8903375488519668 +/- 0.009449783851256728


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

inr_params = count_parameters(inr_seg_model)
classifier_params = count_parameters(classifier_model)

print(f"INR Parameters: {inr_params:,}")
print(f"Classifier Parameters: {classifier_params:,}")


INR Parameters: 661,259
Classifier Parameters: 67,077
